# GONG + all annotated SWEFil + CHASE synthesis — fresh T4 training

Run this single cell in a Colab **T4 GPU** runtime. It starts one fresh ImageNet ConvNeXt-Tiny + U-Net model, using the same 100-epoch optimizer/crop recipe as best84. No best84 or old CHASE model weights initialize this experiment. The old CHASE synthesis and its verified image archive are reused.

- **All 250 annotated SWEFil observations** are admitted, including dates close to the official images. Raw/processed pairs are one observation; processed images are used. The 27 unannotated observations are not treated as negative labels.
- Epochs 4–94 sample GONG 30%, SWEFil 60%, CHASE synthetic 10% (expected 120/240/40 of 400 random crops, respectively; realized counts vary). Epochs 1–3 and 95–100 use GONG only. Each epoch uses 400 image-first crops, rather than a full image pass. The T4 recipe uses 1024 crops, microbatch 1, accumulation 4, FP16, workers 0, AdamW LR 1e-4 and weight decay 0.01.
- The original fixed validation split is retained. Because temporally overlapping SWEFil observations are admitted, this is an exploratory score. Near/far validation diagnostics help distinguish overlap effects from broader improvement.
- The first run downloads pinned SWEFil data and saves a hashed source snapshot to the **new** experiment directory. Run the **same cell** after interruption: valid image caches are reused; completed checkpoints are read back and verified through the Drive API before resume. The interrupted epoch is repeated. Resume uses saved source without fetching GitHub.
- This run keeps immutable checkpoint generations and does not automatically remove older checkpoints or empty Drive Trash. The worker checks Drive API authentication, write/read-back and available storage before training. `auth.authenticate_user()` must use the account mounted as Drive.
- `RUN_UNTIL_EPOCH` is a stage cap, not a changed learning-rate schedule. `PHASE="prepare"` downloads SWEFil and restores existing CHASE synthesis without GPU training. Full logs are appended to `OUTPUT/training.log` and progress is relayed live to Colab.

The result ZIP contains review metadata and metrics, not all image data or model weights. CPU fixture tests cover this launcher; training and Drive authentication must run in Colab. A validation increase does not establish a leaderboard increase.


In [ ]:
# One cell for first launch and resume after interruption or runtime reset.
PHASE = "all"                 # "prepare" downloads/restores data without training.
RUN_UNTIL_EPOCH = 100          # Stage cap; the optimizer schedule always remains 100.
from pathlib import Path
from google.colab import auth, drive, files
import hashlib, json, os, re, signal, subprocess, sys
BASE = Path("/content/drive/MyDrive/filament-segmentation-2026")
OUTPUT = BASE / "fixed_split_experiments/swefil_all_mixed_native_t4_v1"
CHASE_TRAINING_ROOT = BASE / "fixed_split_experiments/chase_shape_best84_recipe_t4_v1"
CACHE_ROOT = Path("/content/swefil_all_mixed_native_t4_cache")
REPOSITORY = "https://github.com/ikjoo5/solar_filament.git"
CHECKOUT = Path("/content/solar_swefil_mixed_github")
drive.mount("/content/drive")
# Drive API verification is required before a checkpoint is committed.
auth.authenticate_user()
OUTPUT = OUTPUT.resolve()
def sha(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()
state_path = OUTPUT / "bootstrap_state.json"
if state_path.is_file():
    state = json.loads(state_path.read_text())
    if state.get("scope") != "swefil_all_mixed_native_t4_v1" or state.get("output_root") != str(OUTPUT):
        raise ValueError("Saved bootstrap belongs to another experiment.")
    version = state.get("version", "")
    if not re.fullmatch("[0-9a-f]{16}", version):
        raise ValueError("Invalid saved source version.")
    snapshot = OUTPUT / "code" / version
    manifest_path = snapshot / "source_manifest.json"
    if manifest_path.is_symlink() or not snapshot.resolve().is_relative_to(OUTPUT) or sha(manifest_path) != state["manifest_sha256"]:
        raise ValueError("Saved source manifest changed; existing files were preserved.")
    manifest = json.loads(manifest_path.read_text())
    identity = json.dumps(manifest, sort_keys=True, separators=(",", ":"), allow_nan=False).encode()
    if hashlib.sha256(identity).hexdigest()[:16] != version:
        raise ValueError("Saved source version differs from its manifest.")
    for name, expected in manifest.items():
        path = snapshot / name
        if path.is_symlink() or not path.resolve().is_relative_to(snapshot.resolve()):
            raise ValueError("Unsafe saved source path.")
        if not path.is_file() or sha(path) != expected:
            raise ValueError("Saved launcher/source changed: " + name)
    bootstrap = snapshot / "tools/swefil_mixed_colab.py"
    if "tools/swefil_mixed_colab.py" not in manifest:
        raise ValueError("Saved manifest does not contain the launcher.")
else:
    if CHECKOUT.exists():
        origin = subprocess.check_output(["git", "-C", str(CHECKOUT), "remote", "get-url", "origin"], text=True).strip()
        dirty = subprocess.check_output(["git", "-C", str(CHECKOUT), "status", "--porcelain"], text=True).strip()
        if origin != REPOSITORY or dirty:
            raise ValueError("Existing checkout differs or has local edits. Preserve it and choose another CHECKOUT.")
        subprocess.run(["git", "-C", str(CHECKOUT), "fetch", "--depth", "1", "origin", "main"], check=True)
        subprocess.run(["git", "-C", str(CHECKOUT), "checkout", "--detach", "FETCH_HEAD"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPOSITORY, str(CHECKOUT)], check=True)
    bootstrap = CHECKOUT / "tools/swefil_mixed_colab.py"
    if not bootstrap.is_file():
        raise FileNotFoundError("The repository does not contain the SWEFil launcher.")
command = [sys.executable, "-u", str(bootstrap), "--base", str(BASE),
           "--output", str(OUTPUT), "--cache-root", str(CACHE_ROOT),
           "--chase-training-root", str(CHASE_TRAINING_ROOT),
           "--phase", PHASE, "--run-until", str(RUN_UNTIL_EPOCH)]
process = subprocess.Popen(command, start_new_session=True,
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, encoding="utf-8", errors="replace", bufsize=1)
try:
    for line in process.stdout:
        print(line, end="", flush=True)
    status = process.wait()
    if status:
        raise subprocess.CalledProcessError(status, command)
finally:
    if process.poll() is None:
        try: os.killpg(process.pid, signal.SIGTERM)
        except ProcessLookupError: pass
        try: process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            try: os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError: pass
            process.wait()
    process.stdout.close()
result = json.loads((OUTPUT / "latest_result.json").read_text())
print("Status:", result.get("status"), "Completed epoch:", result.get("epochs_completed", 0))
if result.get("best_checkpoint_url"):
    print("Verified best weights:", result["best_checkpoint_url"])
if result.get("last_checkpoint_url"):
    print("Verified resumable last weights:", result["last_checkpoint_url"])
if result.get("archive") and result.get("status") in ("complete", "completed", "paused"):
    archive = Path(result["archive"]).resolve()
    if not archive.is_relative_to(OUTPUT) or not archive.is_file() or sha(archive) != result.get("archive_sha256"):
        raise ValueError("Result archive is missing or changed; no download attempted.")
    print("Review ZIP:", archive)
    files.download(str(archive))
else:
    print("Preparation results:", OUTPUT / "latest_result.json")
